# Этап 4: Обучение BERT модели

Fine-tuning rubert-tiny2 для детекции фейковых отзывов

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from train_bert import BERTTrainer

sns.set_style('whitegrid')
%matplotlib inline

## 4.1 Загрузка данных

In [ ]:
df = pd.read_json('../data/processed/reviews_labeled.json')

print(f"Загружено {len(df)} отзывов")
print(f"\nРаспределение классов:")
print(df['label'].value_counts())

# Примеры текстов
print("\nПримеры отзывов:")
print("\nФейк:")
print(df[df['label'] == 1]['text'].iloc[0])
print("\nРеальный:")
print(df[df['label'] == 0]['text'].iloc[0])

## 4.2 Инициализация BERT trainer

Используем `cointegrated/rubert-tiny2` - компактная модель для русского языка

In [ ]:
bert_trainer = BERTTrainer(model_name='cointegrated/rubert-tiny2')

print("✓ BERT trainer инициализирован")
print(f"Модель: {bert_trainer.model_name}")

## 4.3 Подготовка данных

In [ ]:
train_dataset, val_dataset = bert_trainer.prepare_data(df, test_size=0.2)

print(f"Train dataset: {len(train_dataset)} примеров")
print(f"Val dataset: {len(val_dataset)} примеров")

## 4.4 Обучение модели

**Параметры:**
- Эпохи: 5
- Learning rate: 2e-5
- Batch size: 16 (train), 32 (eval)
- Early stopping: 2 эпохи

**Примечание:** Для обучения рекомендуется использовать GPU (Google Colab с GPU runtime)

In [ ]:
# Проверка доступности GPU
import torch
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Обучение (может занять 10-30 минут в зависимости от размера данных)
bert_trainer.train(train_dataset, val_dataset, output_dir='../models/bert')

## 4.5 Оценка модели

In [ ]:
results = bert_trainer.evaluate(val_dataset)

print("\n=== Результаты BERT ===")
for key, value in results.items():
    if not key.startswith('eval_'):
        continue
    metric_name = key.replace('eval_', '')
    print(f"{metric_name}: {value:.4f}")

## 4.6 Примеры предсказаний

In [ ]:
# Тестовые примеры
test_texts = [
    "Отлично! Супер качество, рекомендую!",  # Фейк
    "Товар пришел через неделю. Качество среднее, есть небольшие дефекты на швах. За свою цену нормально.",  # Реальный
    "Огонь! Быстрая доставка, топ!",  # Фейк
    "Использую уже месяц. Сломалась застежка на 3 день, пришлось вернуть. Разочарован.",  # Реальный
]

predictions = bert_trainer.predict(test_texts)

print("\nПредсказания BERT:\n")
for text, prob in zip(test_texts, predictions):
    label = "ФЕЙК" if prob > 0.5 else "РЕАЛЬНЫЙ"
    print(f"Текст: {text}")
    print(f"Вероятность фейка: {prob:.3f} → {label}\n")

## 4.7 Сохранение модели

In [ ]:
bert_trainer.save_model('../models/bert_model')
print("✓ BERT модель сохранена")

## 4.8 Выводы

**Преимущества BERT:**
- Понимает контекст и семантику текста
- Не требует ручной инженерии признаков
- Хорошо работает на коротких текстах

**Недостатки:**
- Требует GPU для обучения
- Медленнее в инференсе чем Random Forest
- Сложнее интерпретировать

**Результаты:**
- ROC-AUC: [будет заполнено после обучения]
- Precision: [будет заполнено]
- Recall: [будет заполнено]
- F1-Score: [будет заполнено]